# Building Model Base Table
This notebook is backbone for understanding of data and assumptions made at the stage of building mbt table.

Purpose: mbt table is ready-to-use for a feautre engineering (encoding, interaction, typecasting, etc) for modeling.

In [1]:
import pandas as pd
import numpy as np
import warnings

pd.set_option('display.max_columns', None)
warnings.filterwarnings("ignore")

In [2]:
%load_ext autoreload
%autoreload 2

from src.utils.load import load
from src.features import constant, sequence, temporal
from src.targets import target

In [3]:
df = load("data/canonical/events.parquet")
schema = load("configs/schema.yaml")
profile = load("configs/feature_profile.yaml")

In [4]:
# percentage of missing values by entries in data
missing_count_by_entries = df.isna().sum().sort_values(ascending=False)
missing_pct_by_entries = (100 * missing_count_by_entries / df.shape[0]).round(2).rename('%missing_by_entries')


# percentage of missing values by cases in data
missing_count_by_cases = df.isna().groupby(df['case_id']).any().sum().sort_values(ascending=False)
missing_pct_by_cases = (100 * missing_count_by_cases / df['case_id'].nunique()).round(2).rename("%missing_by_cases")

missing_pct_record = pd.concat([missing_pct_by_cases, missing_pct_by_entries], axis=1)
missing_pct_record.query("`%missing_by_cases` > 0 or `%missing_by_entries` > 0")

,%missing_by_cases,%missing_by_entries
caused_by_change_id,99.99,99.98
vendor_id,99.94,99.83
asset_id,99.80,99.69
change_request_id,99.61,99.30
root_cause_id,99.04,98.38
assigned_uid,35.15,19.40
reported_symptom,24.58,23.26
assigned_team_gid,10.19,2.95
reported_by_uid,1.01,0.97
resolution_id,0.43,0.50


The first 5 are 99% nulls, for rest we need case level information

caused_by_change_id, vendor_id:
- have 99% missing values, for now they won't be tried for modeling.

asset_id, change_request_id, root_cause_id:
- all these 4 feature are spare, and also have outlier cases with changing values. 
- Later on, flag(presence) of such feature can be tried for modeling.

In [5]:
# handled sparse value with flag 
df.drop(profile['sparse']['columns'], axis=1, inplace=True)

In [6]:
print(f"all {df.columns.size -1} features properties of 24_918 cases at case_level")
grp = df.groupby('case_id')
df_prop = pd.DataFrame(df.dtypes, columns=['dtype']).drop('case_id', axis=0)

df_prop['no_of_changes'] = ((grp.nunique(dropna=True) <= 1).sum() - df.case_id.nunique(dropna=True)).round(2).abs()
df_prop['pct_constant'] = (100* (grp.nunique(dropna=True) <= 1).sum() / df.case_id.nunique(dropna=True)).round(2).abs()

df_prop['all_missing'] = df.isna().groupby(df['case_id']).all().sum()
df_prop['any_missing'] = df.isna().groupby(df['case_id']).any().sum()

df_prop.sort_values(['no_of_changes', 'any_missing'], ascending=True).head(15)

all 32 features properties of 24_918 cases at case_level


,dtype,no_of_changes,pct_constant,all_missing,any_missing
opened_at,datetime64[ns],0,100.00,0,0
created_at,datetime64[ns],0,100.00,0,0
notify_email,boolean,0,100.00,0,0
resolved_at,datetime64[ns],0,100.00,0,0
closed_at,datetime64[ns],0,100.00,0,0
created_at_is_imputed,bool,0,100.00,0,0
affected_uid,object,0,100.00,3,3
location_id,object,0,100.00,6,6
resolved_by_uid,object,0,100.00,99,99
resolution_id,object,0,100.00,107,107




contact_channel:
- 5 out of 25k cases making constant (case_leve) feature changing (event_level).
- assumption: intial contact_channel is used for communication and later channel update.
- The later channels might be used for further communication like escalation, sharing info. Since they're only 5 cases, model won't generalize on them.

In [ ]:
# fix changing record
df['contact_channel'] = grp['contact_channel'].transform('first')

In [ ]:
# dropping minor missing cases
missing_count = df.isna().groupby(df['case_id']).all().sum()
feature_names = missing_count[(missing_count > 0) & (missing_count < 10)].index.tolist()
df = df.dropna(subset=feature_names, axis=0)
df.shape

(137470, 33)

In [66]:
# features missing 100s of cases


In [23]:
df_prop.sort_values(['no_of_changes', 'any_missing'], ascending=True).query('no_of_changes > 10')

,dtype,no_of_changes,pct_constant,all_missing,any_missing
used_knowledge_base,bool,208,99.17,0,0
reopen_count,int64,275,98.90,0,0
urgency_level,object,299,98.80,0,0
impact_level,object,315,98.74,0,0
priority_level,object,383,98.46,0,0
category_id,object,1184,95.25,7,7
reported_symptom,object,1323,94.69,5513,6126
subcategory_id,object,1738,93.03,8,8
assigned_uid,object,2858,88.53,658,8759
met_deadline,bool,9114,63.42,0,0
